# Outer-Open Gomoku — MiniZero Training on Colab

Trains MiniZero (ICGA 2025 gold medalist at Outer-Open Gomoku) on a Colab GPU.

**Order matters:**
1. Mount Drive (cell 1)
2. Setup (cell 2) — installs deps, builds MiniZero (~5 min)
3. Smoke test 30 iters (cell 3) — proves the pipeline before the real run
4. Real run 300 iters (cell 4)
5. Evaluate vs rapfi/slowrenju/pentazen (cell 5)

Checkpoints mirror to Drive every 60s; a recycled VM auto-resumes.
Requires **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [ ]:
# Cell 1 — Mount Google Drive (MUST run first; training refuses to start without it)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Clone repo + setup (deps, MiniZero build with CUDA). ~5 minutes.
# Third-party opponents (rapfi etc.) are optional; failures are skipped.
!rm -rf /content/Outer-Gomoku
!git clone --depth 1 https://github.com/Minh-Nguyen521/outer-gomoku.git /content/Outer-Gomoku
%cd /content/Outer-Gomoku
!MZ=/content/minizero bash tools/colab_minizero.sh setup

In [ ]:
# Cell 3 — SMOKE TEST: 30 iterations. Verifies the whole pipeline
# (self-play + optimization + checkpointing) before committing hours of GPU.
# Expect ~10-20 min. Look for '[Optimization] Finished.' x30 in the output.
!printf 'C\ny\n' | MZ=/content/minizero bash tools/colab_minizero.sh train 30 gmz

In [ ]:
# Cell 4 — REAL RUN: 300 iterations (~3-5 h on T4). Resumable: re-run after
# a VM recycle and it continues from the Drive checkpoint.
# Checkpoints: /content/drive/MyDrive/oog-minizero/gomoku_oo_15x15_gmz_run/model/
!printf 'C\ny\n' | MZ=/content/minizero bash tools/colab_minizero.sh train 300 gmz

In [ ]:
# Cell 5 — Evaluate: MiniZero vs rapfi / slowrenju / pentazen (20 games each).
# Uses the latest checkpoint from Drive. Opponents not built are skipped.
latest = !ls -t /content/drive/MyDrive/oog-minizero/gomoku_oo_15x15_gmz_run/model/*.pt | head -1
print('Using model:', latest[0])
!MZ=/content/minizero bash tools/colab_minizero.sh match {latest[0]}

## Notes
- `MZ=/content/minizero` keeps the build on local disk (compiling/training over Drive's FUSE mount is glacial). Checkpoints still mirror to Drive.
- If the repo on GitHub is stale (uncommitted local changes), push first, or upload `tools/colab_minizero.sh` manually and adjust paths.
- Colab free tier kills idle GPU sessions — keep this tab open or re-run cell 4 to resume.
- Smoke-test model (30 iters) is weak; expect it to lose badly to rapfi. It only proves the pipeline works.